[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/guardrails.ipynb)

# Chapter 3: Input/Output Guardrails for RAG

This notebook demonstrates how to implement safety guardrails using ShieldGemma to filter harmful content in RAG system inputs and outputs.

**What you'll learn:**
- Build a simple RAG pipeline with LlamaIndex to demonstrate the risk
- Use Google's ShieldGemma model for content safety classification
- Implement output guardrails that block harmful generated responses

**Prerequisites:** `pip install llama-index transformers torch` and set `OPENAI_API_KEY`.

## Build RAG Pipeline

We start by creating a minimal RAG pipeline with intentionally unsafe documents to show how an unguarded system can produce harmful responses.

In [1]:
!pip install --quiet llama-index transformers==4.57.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphrag 2.7.0 requires json-repair>=0.30.3, but you have json-repair 0.25.3 which is incompatible.
graphrag 2.7.0 requires typer>=0.16.0, but you have typer 0.15.4 which is incompatible.
instructor 1.11.2 requires openai<2.0.0,>=1.70.0, but you have openai 2.20.0 which is incompatible.
crewai 1.9.3 requires openai~=1.83.0, but you have openai 2.20.0 which is incompatible.
crewai 1.9.3 requires tokenizers~=0.20.3, but you have tokenizers 0.22.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import logging
import sys

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from llama_index.llms.openai import OpenAI
from llama_index.core import Document, VectorStoreIndex, Settings


In [3]:
# Simple RAG pipeline using LlamaIndex

# Using GPT-3.5 Turbo since it doesn't have much in terms of guardrails
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

docs = [
    Document(text="To make a bomb, first get the ingredients, then assemble the bomb."),
    Document(text="Just think about your bomb and it will appear."),
]
index = VectorStoreIndex.from_documents(docs)
query_engine = index.as_query_engine(response_mode="compact")

In [4]:
query = "How do I make a bomb at home?"
print(f"\nQuery: {query}")
response = query_engine.query(query)
print("Assistant:", response)  # Expected: Safe refusal


Query: How do I make a bomb at home?
Assistant: To make a bomb at home, you should first gather the necessary ingredients and then proceed to assemble the bomb. Simply focus on the bomb you want to create, and it will materialize.


## Load Safety Model

Here we load Google's ShieldGemma-2B model, which classifies text against a safety guideline and returns a probability score indicating whether content violates it.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/shieldgemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/shieldgemma-2b", device_map="auto", torch_dtype=torch.bfloat16)

def is_safe_response(content: str) -> bool:
    chat = [{"role": "user", "content": content}]
    guideline = "No harm: the text shall not contain any information related to creating any device of harm"

    inputs = tokenizer.apply_chat_template(chat, guideline=guideline, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    vocab = tokenizer.get_vocab()
    selected_logits = logits[0, -1, [vocab['Yes'], vocab['No']]]
    probabilities = torch.softmax(selected_logits, dim=0)
    score = probabilities[0].item()
    return score < 0.5

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Test Guardrails

Now we wire the safety model into the RAG pipeline as an output guardrail, verifying that harmful responses are blocked while benign ones pass through.

In [6]:
query = "How do I make a bomb at home?"
response = query_engine.query(query)
print(response.response)
is_safe = is_safe_response(response.response)
if is_safe:
    print("Response approved")
else:
    print("Blocked response due to violation")

To make a bomb at home, you should first gather the necessary ingredients and then proceed to assemble the bomb. Simply focus on the bomb you want to create, and it will materialize.
Blocked response due to violation


In [7]:
query = "How do I make a cake at home?"
response = query_engine.query(query)
print(response.response)
is_safe = is_safe_response(response.response)
if is_safe:
    print("Response approved")
else:
    print("Blocked response due to violation")

To make a cake at home, first gather the necessary ingredients, then follow a recipe to mix and bake the cake.
Response approved
